# Gradient Descent — Exercises

Companion to the note [Gradient Descent](Gradient%20Descent.md).

Practise the update $\theta_{t+1}=\theta_t-\eta\nabla L(\theta_t)$ and its variants (batch, SGD, mini-batch), the stability limit $\eta<2/\lambda_{\max}(H)$, momentum, learning-rate schedules, convergence rates and the debugging habits from the note (gradient checks, line search): by hand and from scratch in NumPy.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×5 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.optimize import minimize, approx_fprime
from scipy.special import expit
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
# shared regression data: y = X w_true + noise
X_reg = rng.normal(size=(500, 3))
w_true_reg = np.array([2.0, -1.0, 0.5])
y_reg = X_reg @ w_true_reg + 0.3 * rng.normal(size=500)
w_ls = np.linalg.lstsq(X_reg, y_reg, rcond=None)[0]   # exact least-squares solution

## Part A · Concepts

### Exercise 1 · Batch, stochastic, mini-batch
*🧠 Concept · ★☆☆*

For a dataset of $N=10^6$ samples, compare batch GD, SGD ($B=1$) and mini-batch ($B=256$) on: cost per update, cost per
epoch, noise of the gradient estimate, use of GPU parallelism, and number of updates per epoch. Why is mini-batch the standard?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Per-epoch cost is the same ($N$ gradient evaluations) for all three; what differs is how many updates you get and how noisy each is.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| | Batch | SGD | Mini-batch (256) |
|---|---|---|---|
| cost per update | $N$ | $1$ | $256$ |
| updates per epoch | $1$ | $10^6$ | $\approx 3900$ |
| gradient noise | none | high | variance $\propto 1/B$ |
| GPU use | good but one slow step | poor (no parallelism) | good |

Mini-batch gives many updates per epoch (fast progress) while each step is vectorised and much less noisy than SGD. A bit of
noise also helps escape saddles and sharp minima.

</details>

### Exercise 2 · Reading loss curves
*🧠 Concept · ★☆☆*

Diagnose each training-loss curve and suggest a fix:

(a) The loss goes to `nan` after a few steps. (b) The loss decreases but extremely slowly and nearly linearly.
(c) The loss drops fast, then bounces around a plateau well above the training loss you expect.
(d) The loss zig-zags strongly from step to step but trends downward.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about the learning rate relative to $2/\lambda_{\max}$, and about gradient noise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Divergence: $\eta$ too large (or exploding gradients). Lower $\eta$, add warm-up, clip gradients, check the data for infs.
(b) $\eta$ too small (or badly scaled features). Increase $\eta$, standardise inputs, use momentum/Adam.
(c) Constant-$\eta$ SGD noise floor: decay the learning rate (step/cosine schedule) or increase the batch size.
(d) Oscillation across a ravine: $\eta$ close to the stability limit in the steepest direction. Use momentum, which averages
out the zig-zag, or a slightly smaller $\eta$. See [[Training Tricks]].

</details>

### Exercise 3 · Why SGD needs a decaying learning rate
*🧠 Concept · ★★☆*

The note says "SGD needs decaying $\eta$". Explain why constant-$\eta$ SGD does not converge to the exact minimiser even on a
convex problem, while batch GD does. State the Robbins–Monro conditions on $\eta_t$ and check whether $\eta_t = 1/t$ and
$\eta_t=1/\sqrt t$ satisfy them.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

At the optimum $\nabla L(\theta^*)=0$, but a single-sample gradient $\nabla\ell_i(\theta^*)$ is generally not zero.

</details>

<details>
<summary><b>✅ Solution</b></summary>

At $\theta^*$ the full gradient vanishes, so batch GD stops there. Individual sample gradients do not vanish, so SGD keeps being
kicked around with step size $\eta\lVert\nabla\ell_i\rVert$: it hovers in a ball whose radius scales with $\eta$ times the gradient variance.
Robbins–Monro: $\sum_t\eta_t=\infty$ (can travel arbitrarily far) and $\sum_t\eta_t^2<\infty$ (noise eventually averages out).
$1/t$ satisfies both. $1/\sqrt t$ has $\sum 1/t = \infty$ for the squares, so it violates the second condition, though it is
often used with iterate averaging.

</details>

### Exercise 4 · A debugging checklist
*🧠 Concept · ★★☆*

Your new model's training loss does not decrease. Following the note's debugging advice, list in order the checks you would
perform and what each one rules out.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The note mentions three: overfit a tiny batch, check the loss goes down, gradient-check numerically.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Sanity-check the initial loss** (e.g. $\ln K$ for $K$ balanced classes): rules out wrong labels/loss wiring.
2. **Gradient check** on a tiny model with finite differences: rules out bugs in the backward pass.
3. **Overfit a tiny batch** (e.g. 10 examples) to near-zero loss: if it cannot, the bug is in model/optimiser code, not in data size or regularisation.
4. **Learning-rate sweep** (log scale, e.g. $10^{-5}\dots1$): rules out a bad $\eta$.
5. Check inputs: scaling, NaNs, shuffled labels, data loader bugs.
Only then tune architecture or regularisation ([[Training Tricks]]).

</details>

## Part B · By hand

### Exercise 5 · Two steps by hand
*✍️ By hand · ★☆☆*

Minimise $L(\theta) = (\theta-3)^2$ from $\theta_0 = 0$ with $\eta=0.1$. Compute $\theta_1$ and $\theta_2$. Then show in general that
$\theta_t - 3 = (1-2\eta)^t(\theta_0-3)$ and give the range of $\eta$ for which GD converges.

In [ ]:
theta1 = None
theta2 = None
eta_upper = None   # GD converges iff 0 < eta < eta_upper

_t = 0.0
for _ in range(2):
    _t = _t - 0.1 * 2 * (_t - 3)
    if _ == 0: _t1 = _t
check('theta_1', theta1, _t1); check('theta_2', theta2, _t)
check("eta upper bound = 2 / L''", eta_upper, 2 / 2)

<details>
<summary><b>💡 Hint</b></summary>

$L'(\theta) = 2(\theta-3)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\theta_1 = 0 - 0.1\cdot2(0-3) = 0.6$, $\theta_2 = 0.6 - 0.1\cdot 2(0.6-3) = 0.6+0.48 = 1.08$.
$\theta_{t+1}-3 = \theta_t - 3 - 2\eta(\theta_t-3) = (1-2\eta)(\theta_t-3)$. Convergence iff $|1-2\eta|<1 \iff 0<\eta<1 = 2/L''$,
the note's $2/\lambda_{\max}$ rule. At $\eta=0.5$ it converges in a single step; for $0.5<\eta<1$ it oscillates but converges.

```python
theta1 = 0.6
theta2 = 1.08
eta_upper = 1.0
```

</details>

### Exercise 6 · Stability limit and the optimal fixed step
*✍️ By hand · ★★☆*

For $L(x,y) = \tfrac12(10x^2 + y^2)$: (a) what is the largest stable learning rate? (b) The step minimising the worst-case
contraction factor $\max_i|1-\eta\lambda_i|$ is $\eta^*=2/(\lambda_{\min}+\lambda_{\max})$; compute it and the resulting factor
$\frac{\kappa-1}{\kappa+1}$ with $\kappa=\lambda_{\max}/\lambda_{\min}$.

In [ ]:
eta_max = None
eta_opt = None
rate_opt = None

_lam = np.linalg.eigvalsh(np.diag([10., 1.]))
check('eta_max', eta_max, 2 / _lam.max())
check('eta_opt', eta_opt, 2 / (_lam.min() + _lam.max()))
_e = 2 / (_lam.min() + _lam.max())
check('contraction', rate_opt, np.max(np.abs(1 - _e * _lam)))

<details>
<summary><b>💡 Hint</b></summary>

$H=\mathrm{diag}(10,1)$. Each coordinate is multiplied by $(1-\eta\lambda_i)$ per step.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\eta<2/10 = 0.2$. (b) $\eta^* = 2/11\approx0.182$, giving $|1-\tfrac{20}{11}| = |1-\tfrac{2}{11}| = \tfrac{9}{11}\approx0.818$ per step,
which equals $\frac{\kappa-1}{\kappa+1}$ with $\kappa=10$. The step is dictated by the steepest direction, while progress in the
flattest direction is slow: ill-conditioning.

```python
eta_max = 0.2
eta_opt = 2 / 11
rate_opt = 9 / 11
```

</details>

### Exercise 7 · Momentum as an exponential sum
*✍️ By hand · ★★☆*

With $v_0=0$, $v\leftarrow\beta v+\nabla L$, $\theta\leftarrow\theta-\eta v$: write $v_t$ as a weighted sum of past gradients.
If the gradient is constant $g$ (a long gentle slope), what does $v_t$ converge to? With $\beta=0.9$, by what factor is the
effective step size amplified? Store that factor.

In [ ]:
amplification = None

_v = 0.0
for _ in range(2000):
    _v = 0.9 * _v + 1.0
check('effective step factor', amplification, _v)

<details>
<summary><b>💡 Hint</b></summary>

Unroll: $v_t = g_t + \beta g_{t-1} + \beta^2 g_{t-2}+\dots$. Geometric series.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$v_t = \sum_{k=1}^{t}\beta^{t-k}g_k$. For constant $g$: $v_t = g\frac{1-\beta^t}{1-\beta}\to\frac{g}{1-\beta}$, so the step becomes
$\frac{\eta}{1-\beta}g$, $10\times$ larger for $\beta=0.9$. Along ravines, oscillating gradient components cancel in the sum while the
consistent component accumulates: momentum accelerates along the valley floor and damps the zig-zag.

```python
amplification = 10.0
```

</details>

### Exercise 8 · Linear versus sublinear rates
*✍️ By hand · ★★☆*

(a) A strongly convex problem contracts the error by a factor $0.9$ per iteration (linear rate). How many iterations reduce the
error by a factor $10^{6}$? (b) A convex $L$-smooth problem has error $\le C/t$. If after 10 iterations the error is $0.1$,
how many iterations does the bound need to reach $10^{-4}$?

In [ ]:
iters_linear = None
iters_sublinear = None

check('linear rate', iters_linear, next(t for t in range(1, 10**4) if 0.9 ** t <= 1e-6))
check('O(1/t) rate', iters_sublinear, next(t for t in range(1, 10**6) if 1.0 / t <= 1e-4 + 1e-15))

<details>
<summary><b>💡 Hint</b></summary>

(a) Solve $0.9^t\le10^{-6}$. (b) $C = 0.1\cdot10 = 1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $t\ge\ln(10^{-6})/\ln0.9 = 13.8155/0.10536\approx131.1$, so $132$ iterations; every 22 iterations add one correct digit.
(b) $C=1$, $1/t\le10^{-4}\Rightarrow t=10\,000$. Each extra digit costs $10\times$ more iterations: that is why strong convexity
(e.g. adding an L2 penalty) speeds optimisation up so much.

```python
iters_linear = 132
iters_sublinear = 10_000
```

</details>

### Exercise 9 · Mini-batch gradients are unbiased
*✍️ By hand · ★★☆*

Let $L(\theta)=\frac1N\sum_{i=1}^N\ell_i(\theta)$ and let $\hat g = \frac1B\sum_{i\in\mathcal B}\nabla\ell_i$ with $\mathcal B$ drawn
uniformly **with replacement**. Show $\mathbb E[\hat g]=\nabla L$ and $\mathrm{Var}(\hat g) = \frac{1}{B}\mathrm{Var}(\nabla\ell_I)$
(per coordinate, $I$ uniform). What does quadrupling the batch size do to the gradient's standard deviation?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Each sampled index $I_j$ is uniform on $\{1,\dots,N\}$, so $\mathbb E[\nabla\ell_{I_j}] = \frac1N\sum_i\nabla\ell_i$. Draws are independent.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbb E[\hat g] = \frac1B\sum_{j=1}^B\mathbb E[\nabla\ell_{I_j}] = \frac1B\cdot B\cdot\nabla L = \nabla L$.
Independence gives $\mathrm{Var}(\hat g) = \frac1{B^2}\sum_j\mathrm{Var}(\nabla\ell_{I_j}) = \frac{\sigma^2}{B}$.
Quadrupling $B$ halves the standard deviation, at $4\times$ the compute: large batches have diminishing returns, which is one reason
for the linear "scale $\eta$ with $B$" heuristic (and its limits).

</details>

## Part C · Code from scratch

### Exercise 10 · A generic gradient-descent loop
*💻 Code · ★☆☆*

Implement `gd(grad, x0, lr, steps)` returning the whole trajectory as an array of shape `(steps+1, d)`. Use it to minimise
$f(x)=\tfrac12x^\top Ax - b^\top x$ with $A=\begin{pmatrix}3&1\\1&2\end{pmatrix}$, $b=(1,1)$ for 200 steps with $\eta=0.2$, and plot
the trajectory over the contour lines.

In [ ]:
def gd(grad, x0, lr, steps):
    # TODO
    return None

A_q = np.array([[3., 1.], [1., 2.]]); b_q = np.array([1., 1.])
traj = gd(lambda x: A_q @ x - b_q, np.array([2., -2.]), 0.2, 200)

check('trajectory shape', None if traj is None else traj.shape, (201, 2))
check('final iterate = A^-1 b', None if traj is None else traj[-1], np.linalg.solve(A_q, b_q))

<details>
<summary><b>💡 Hint</b></summary>

Preallocate `xs = np.zeros((steps+1, len(x0)))`, set `xs[0] = x0` and fill in a loop.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\lambda_{\max}(A)\approx3.62$, so $\eta=0.2<2/3.62\approx0.55$ is stable. The path bends towards the long axis of the elliptical contours, the typical GD path on an ill-conditioned quadratic.

```python
def gd(grad, x0, lr, steps):
    xs = np.zeros((steps + 1, len(x0))); xs[0] = x0
    for t in range(steps):
        xs[t + 1] = xs[t] - lr * grad(xs[t])
    return xs

A_q = np.array([[3., 1.], [1., 2.]]); b_q = np.array([1., 1.])
traj = gd(lambda x: A_q @ x - b_q, np.array([2., -2.]), 0.2, 200)
g1, g2 = np.meshgrid(np.linspace(-2.5, 2.5, 100), np.linspace(-2.5, 2.5, 100))
F = 0.5 * (A_q[0, 0] * g1 ** 2 + 2 * A_q[0, 1] * g1 * g2 + A_q[1, 1] * g2 ** 2) - b_q[0] * g1 - b_q[1] * g2
plt.contour(g1, g2, F, 20); plt.plot(traj[:, 0], traj[:, 1], 'o-', ms=3); plt.gca().set_aspect('equal'); plt.show()
```

</details>

### Exercise 11 · Finding the divergence threshold empirically
*💻 Code · ★★☆*

For the least-squares loss $L(w)=\frac1N\lVert Xw-y\rVert^2$ on `X_reg, y_reg` (setup cell), the Hessian is $H=\frac2NX^\top X$.
Find by bisection the largest learning rate for which 3000 GD steps from $w=0$ do **not** blow up (final loss below the initial
loss). Store it in `eta_crit` and compare with $2/\lambda_{\max}(H)$.

In [ ]:
def diverges(lr, steps=3000):
    # TODO: run GD on the MSE and return True if the final loss exceeds the initial loss
    return None

eta_crit = None

check('empirical threshold = 2 / lambda_max', eta_crit, 2 / np.linalg.eigvalsh(2 / len(y_reg) * X_reg.T @ X_reg).max(), tol=5e-3)

<details>
<summary><b>💡 Hint</b></summary>

Bisection between `lo=0.01` (stable) and `hi=5` (diverges): 40 halvings give plenty of precision.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Just above the threshold the error grows like $|1-\eta\lambda_{\max}|^t$, so with 3000 steps even a tiny excess eventually explodes and the bisection pins $2/\lambda_{\max}$ down to a fraction of a percent. Larger $\lambda_{\max}$ (unscaled features) means a smaller usable $\eta$.

```python
def diverges(lr, steps=3000):
    w = np.zeros(3); N = len(y_reg)
    loss0 = np.mean((X_reg @ w - y_reg) ** 2)
    for _ in range(steps):
        w = w - lr * 2 / N * X_reg.T @ (X_reg @ w - y_reg)
        if not np.all(np.abs(w) < 1e6):
            return True
    return np.mean((X_reg @ w - y_reg) ** 2) > loss0

lo, hi = 0.01, 5.0
for _ in range(40):
    mid = (lo + hi) / 2
    lo, hi = (lo, mid) if diverges(mid) else (mid, hi)
eta_crit = lo
print(eta_crit, 2 / np.linalg.eigvalsh(2 / len(y_reg) * X_reg.T @ X_reg).max())
```

</details>

### Exercise 12 · Mini-batch SGD for linear regression
*💻 Code · ★★☆*

Implement `minibatch_sgd(X, y, B, lr, epochs, seed)` for the MSE: shuffle once per epoch, step on each mini-batch. Use a
learning rate that decays as $\eta_t = \eta_0/(1 + t/100)$ (with $t$ the update count). Run with $B=32$, $\eta_0=0.1$, 30 epochs
and compare with the exact solution `w_ls`.

In [ ]:
def minibatch_sgd(X, y, B=32, lr=0.1, epochs=30, seed=0):
    # TODO
    return None

w_sgd = minibatch_sgd(X_reg, y_reg)

check('SGD close to least squares', w_sgd, w_ls, tol=1e-2)

<details>
<summary><b>💡 Hint</b></summary>

`perm = r.permutation(N)`; for `i in range(0, N, B)`: `idx = perm[i:i+B]`, gradient $\frac{2}{|idx|}X_{idx}^\top(X_{idx}w-y_{idx})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Shuffling each epoch makes mini-batches unbiased samples (sampling without replacement within an epoch). Without the decay the iterates hover around `w_ls` at a distance set by $\eta$ (next exercise).

```python
def minibatch_sgd(X, y, B=32, lr=0.1, epochs=30, seed=0):
    r = np.random.default_rng(seed)
    N, d = X.shape; w = np.zeros(d); t = 0
    for _ in range(epochs):
        perm = r.permutation(N)
        for i in range(0, N, B):
            idx = perm[i:i + B]
            g = 2 / len(idx) * X[idx].T @ (X[idx] @ w - y[idx])
            w -= lr / (1 + t / 100) * g; t += 1
    return w

w_sgd = minibatch_sgd(X_reg, y_reg)
```

</details>

### Exercise 13 · The SGD noise floor
*💻 Code · ★★★*

Run single-sample SGD ($B=1$) on `X_reg, y_reg` for $20\,000$ updates from $w=0$, once with constant $\eta=0.02$ and once with
$\eta_t = 0.02/(1+t/500)$. For each run record $\lVert w_t - w_{ls}\rVert$ and store the mean over the last 2000 updates in
`err_const` and `err_decay`. Plot both curves on a log scale.

In [ ]:
err_const = None
err_decay = None

check('decaying lr gets much closer than constant', None if err_const is None else err_decay < err_const / 3, True)

<details>
<summary><b>💡 Hint</b></summary>

Draw the indices for both runs from the same seed so the comparison is fair: `idx = np.random.default_rng(1).integers(0, N, 20000)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Constant-$\eta$ SGD plateaus at an error proportional to $\sqrt\eta$ times the gradient noise; the decaying schedule keeps shrinking the error (Exercise 3). In deep learning, step/cosine schedules play the same role.

```python
N = len(y_reg); idx = np.random.default_rng(1).integers(0, N, 20000)
curves = {}
for name, sched in [('const', lambda t: 0.02), ('decay', lambda t: 0.02 / (1 + t / 500))]:
    w = np.zeros(3); errs = []
    for t, i in enumerate(idx):
        w = w - sched(t) * 2 * X_reg[i] * (X_reg[i] @ w - y_reg[i])
        errs.append(np.linalg.norm(w - w_ls))
    curves[name] = errs
err_const = np.mean(curves['const'][-2000:]); err_decay = np.mean(curves['decay'][-2000:])
plt.semilogy(curves['const'], label='constant'); plt.semilogy(curves['decay'], label='decaying'); plt.legend(); plt.show()
print(err_const, err_decay)
```

</details>

### Exercise 14 · Momentum on a ravine
*💻 Code · ★★☆*

On $f(x)=\tfrac12x^\top\mathrm{diag}(1,100)x$ from $x_0=(1,1)$, count iterations until $\lVert x\rVert<10^{-6}$ for
(a) plain GD with $\eta = 2/(1+100)$ and (b) momentum in the note's form ($v\leftarrow\beta v+\nabla f$, $x\leftarrow x-\eta v$) with
the heavy-ball tuning $\eta = \frac{4}{(\sqrt{L}+\sqrt{\mu})^2}$, $\beta = \left(\frac{\sqrt\kappa-1}{\sqrt\kappa+1}\right)^2$ ($L=100,\mu=1$).
Store `it_plain` and `it_mom`.

In [ ]:
it_plain = None
it_mom = None

_k = 100
_pred_plain = int(np.ceil(np.log(1e-6 / np.sqrt(2)) / np.log((_k - 1) / (_k + 1))))
check('plain GD matches theory (within 2)', None if it_plain is None else abs(it_plain - _pred_plain) <= 2, True)
check('momentum is at least 5x faster', None if it_mom is None else it_mom * 5 < it_plain, True)

<details>
<summary><b>💡 Hint</b></summary>

Write a loop with a counter and a safety cap (e.g. 10 000 iterations).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Plain GD needs about 700 iterations (rate $\frac{\kappa-1}{\kappa+1}=0.98$), heavy-ball momentum about 100 (rate $\approx\frac{\sqrt\kappa-1}{\sqrt\kappa+1}=0.82$): the dependence on $\kappa$ becomes $\sqrt\kappa$. Adaptive methods ([[Optimizers]]) attack the same problem by rescaling coordinates.

```python
lam = np.array([1., 100.])
x = np.ones(2); it_plain = 0
while np.linalg.norm(x) >= 1e-6 and it_plain < 10_000:
    x = x - 2 / 101 * lam * x; it_plain += 1
eta = 4 / (np.sqrt(100) + 1) ** 2; beta = ((10 - 1) / (10 + 1)) ** 2
x = np.ones(2); v = np.zeros(2); it_mom = 0
while np.linalg.norm(x) >= 1e-6 and it_mom < 10_000:
    v = beta * v + lam * x; x = x - eta * v; it_mom += 1
print(it_plain, it_mom)
```

</details>

### Exercise 15 · Learning-rate schedules
*💻 Code · ★☆☆*

Implement the schedules from the note as functions of the step $t$:

- `step_decay(t, eta0, drop=0.5, every=10)`: multiply by `drop` every `every` steps.
- `cosine(t, T, eta0, eta_min=0)`: $\eta_{\min} + \tfrac12(\eta_0-\eta_{\min})(1+\cos(\pi t/T))$.
- `warmup_cosine(t, T, warmup, eta0)`: linear from $0$ to $\eta_0$ over `warmup` steps, then cosine over the remaining $T-$`warmup`.

Plot all three for $T=100$.

In [ ]:
def step_decay(t, eta0, drop=0.5, every=10):
    return None
def cosine(t, T, eta0, eta_min=0.0):
    return None
def warmup_cosine(t, T, warmup, eta0):
    return None

check('step decay at t=25', step_decay(25, 1.0), 0.25)
check('cosine endpoints and midpoint', None if cosine(0, 100, 1.0) is None else [cosine(0, 100, 1.0, 0.1), cosine(50, 100, 1.0, 0.1), cosine(100, 100, 1.0, 0.1)], [1.0, 0.55, 0.1])
check('warm-up: half way, peak, end', None if warmup_cosine(5, 100, 10, 1.0) is None else [warmup_cosine(5, 100, 10, 1.0), warmup_cosine(10, 100, 10, 1.0), warmup_cosine(100, 100, 10, 1.0)], [0.5, 1.0, 0.0])

<details>
<summary><b>💡 Hint</b></summary>

`step_decay`: `eta0 * drop ** (t // every)`. For the warm-up part of `warmup_cosine` return `eta0 * t / warmup`; afterwards call `cosine(t - warmup, T - warmup, eta0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Warm-up avoids huge early updates while adaptive optimiser statistics and normalisation layers are still unreliable; cosine decay then anneals smoothly to a small $\eta$, removing the SGD noise floor.

```python
def step_decay(t, eta0, drop=0.5, every=10):
    return eta0 * drop ** (t // every)
def cosine(t, T, eta0, eta_min=0.0):
    return eta_min + 0.5 * (eta0 - eta_min) * (1 + np.cos(np.pi * t / T))
def warmup_cosine(t, T, warmup, eta0):
    if t < warmup:
        return eta0 * t / warmup
    return cosine(t - warmup, T - warmup, eta0)

ts = np.arange(101)
plt.plot(ts, [step_decay(t, 1.0) for t in ts], label='step')
plt.plot(ts, [cosine(t, 100, 1.0) for t in ts], label='cosine')
plt.plot(ts, [warmup_cosine(t, 100, 10, 1.0) for t in ts], label='warm-up + cosine'); plt.legend(); plt.show()
```

</details>

### Exercise 16 · Gradient check for logistic regression
*💻 Code · ★★☆*

Implement `logloss(w, X, y)` (mean binary cross-entropy, $y\in\{0,1\}$) and its analytic gradient `logloss_grad(w, X, y)`
$=\frac1N X^\top(\sigma(Xw)-y)$. Compare with a central-difference gradient and store the relative error
$\frac{\lVert g_a-g_n\rVert}{\lVert g_a\rVert+\lVert g_n\rVert}$ in `rel_err`. A correct gradient gives about $10^{-9}$ or smaller.

In [ ]:
def logloss(w, X, y):
    return None
def logloss_grad(w, X, y):
    return None

X_c = rng.normal(size=(50, 4)); y_c = (rng.random(50) < 0.5).astype(float); w_c = rng.normal(size=4)
rel_err = None

check('analytic gradient', logloss_grad(w_c, X_c, y_c), approx_fprime(w_c, lambda w: np.mean(np.logaddexp(0, X_c @ w) - y_c * (X_c @ w)), 1e-7), tol=1e-5)
check('relative error tiny', None if rel_err is None else rel_err < 1e-6, True)

<details>
<summary><b>💡 Hint</b></summary>

Stable loss: $\ell_i = \log(1+e^{z_i}) - y_iz_i$ with $z=Xw$ (`np.logaddexp(0, z)`).

</details>

<details>
<summary><b>✅ Solution</b></summary>

The relative error is scale-free; values around $10^{-2}$ or larger mean a bug. Always gradient-check on small, random, non-symmetric inputs, in float64.

```python
def logloss(w, X, y):
    z = X @ w
    return np.mean(np.logaddexp(0, z) - y * z)
def logloss_grad(w, X, y):
    return X.T @ (expit(X @ w) - y) / len(y)

X_c = rng.normal(size=(50, 4)); y_c = (rng.random(50) < 0.5).astype(float); w_c = rng.normal(size=4)
g_a = logloss_grad(w_c, X_c, y_c)
g_n = np.array([(logloss(w_c + e, X_c, y_c) - logloss(w_c - e, X_c, y_c)) / 2e-6 for e in 1e-6 * np.eye(4)])
rel_err = np.linalg.norm(g_a - g_n) / (np.linalg.norm(g_a) + np.linalg.norm(g_n))
print(rel_err)
```

</details>

### Exercise 17 · Backtracking line search
*💻 Code · ★★★*

Instead of a fixed $\eta$, implement GD with **Armijo backtracking**: start each step with $\eta=1$ and halve it until
$L(w-\eta g)\le L(w) - c\,\eta\lVert g\rVert^2$ with $c=10^{-4}$. Apply it to the L2-regularised logistic loss
$L(w) = \text{logloss}(w) + \frac{\lambda}2\lVert w\rVert^2$ ($\lambda=0.01$) on `X_c, y_c` for 500 iterations and compare the final
loss with the L-BFGS optimum.

In [ ]:
lam_bt = 0.01
def reg_loss(w):
    return None   # TODO: logloss + lam/2 ||w||^2
def reg_grad(w):
    return None

loss_bt = None   # final loss after 500 backtracking GD steps

_f = lambda w: np.mean(np.logaddexp(0, X_c @ w) - y_c * (X_c @ w)) + lam_bt / 2 * w @ w
check('backtracking reaches the optimum', loss_bt, minimize(_f, np.zeros(4), method='L-BFGS-B', options={'gtol': 1e-10}).fun, tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

Inner loop: `while reg_loss(w - eta * g) > reg_loss(w) - c * eta * g @ g: eta /= 2`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The Armijo condition guarantees sufficient decrease every step without knowing the smoothness constant; it automatically finds $\eta\approx1/L$. Line search is standard in full-batch methods (L-BFGS uses a Wolfe line search) but too costly and noisy for SGD.

```python
lam_bt = 0.01
def reg_loss(w):
    return logloss(w, X_c, y_c) + lam_bt / 2 * w @ w
def reg_grad(w):
    return logloss_grad(w, X_c, y_c) + lam_bt * w

w = np.zeros(4)
for _ in range(500):
    g = reg_grad(w); eta = 1.0
    while reg_loss(w - eta * g) > reg_loss(w) - 1e-4 * eta * g @ g:
        eta /= 2
    w = w - eta * g
loss_bt = reg_loss(w)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Gradient Descent](Gradient%20Descent.md).*